In [7]:
import torch
import os
import shutil
import torchvision
from torchvision.transforms.functional import InterpolationMode
from torchvision.io.image import ImageReadMode
import matplotlib.pyplot as plt
import tqdm

In [8]:
device:str = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [9]:
def resize_images(input_path:str, output_path:str, size:tuple[int, int]) -> None:
    if os.path.isdir(output_path):
        shutil.rmtree(output_path)
    os.makedirs(output_path) # Changed to makedirs for safety

    raw_images_path: list[str] = os.listdir(input_path)
    
    # Combined transform: 
    # 1. Resize the shortest side to match the target 'size' while keeping ratio
    # 2. Crop the center to ensure the final output is exactly size x size
    transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize(max(size), interpolation=InterpolationMode.BICUBIC, antialias=True),
        torchvision.transforms.CenterCrop(size)
    ])

    desc = f"Resizing & Cropping to ({size[0]}, {size[1]})"
    with tqdm.tqdm(total=len(raw_images_path), desc=desc) as pbar:
        for i, image_local_path in enumerate(raw_images_path):
            image_absolute_path: str = os.path.join(input_path, image_local_path)
            
            try:
                # Load image
                raw_image: torch.Tensor = torchvision.io.read_image(
                    image_absolute_path, 
                    torchvision.io.ImageReadMode.RGB
                ).to(device)
                
                # Apply transforms (Resize + Crop)
                output_image: torch.Tensor = transform(raw_image).cpu() 
                
                # Save
                output_filename = os.path.join(output_path, f"{i}.png")
                torchvision.io.write_png(output_image, output_filename, compression_level=0)
                
            except Exception as e:
                print(f"Error processing {image_local_path}: {e}")
                
            pbar.update(1)

input_path:str = os.path.join(os.getcwd(), "Dataset", "raw")
output_paths:list[str] = [
    # os.path.join(os.getcwd(), "Dataset", "64x64"),
    os.path.join(os.getcwd(), "Dataset", "128x128"),
    # os.path.join(os.getcwd(), "Dataset", "256x256"),
    # os.path.join(os.getcwd(), "Dataset", "512x512")
]
new_sizes:list[tuple[int, int]] = [
    # (64, 64),
    (128, 128),
    # (256, 256),
    # (512, 512),
]

assert len(output_paths) == len(new_sizes)

for i in range(0, len(output_paths)):
    resize_images(input_path, output_paths[i], new_sizes[i])

Resizing & Cropping to (128, 128): 100%|██████████| 13059/13059 [00:26<00:00, 497.83it/s]
